In [1]:
from openai import OpenAI
import json
import os
import re
import random
import itertools

In [2]:
# Initialize the OpenAI client with API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # Specified value was saved
if client.api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable not set")


In [3]:
# Load profiles and prompts from JSON files
with open("C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data generation/personas generation/profile.json", "r", encoding="UTF-8") as file:
    profile = json.load(file)

with open("C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data generation/personas generation/prompts.json", "r", encoding="UTF-8") as file:
    prompts = json.load(file)
  

In [4]:
# Separate names by gender and ethnicity for correct association
names_by_gender = {'Male': ["Mohammed", "Abdullah", "Rudolf", "Johan", "Ali", "Mehmet", "Pieter", "Jan"], 'Female': ["Aicha", "Fatima", "Julia", "Ingrid", "Ayşe", "Fatma", "Johanna", "Maria"]}
names_by_ethnicity = {'Moroccan': ["Mohammed", "Abdullah", "Aicha", "Fatima"], 'Surinamese': ["Rudolf", "Johan", "Julia", "Ingrid"], 'Turkish': ["Ali", "Mehmet", "Ayşe", "Fatma"], 'Dutch': ["Pieter", "Jan", "Johanna", "Maria"]}

# Generate all combinations respecting gender-specific and ethnicity-specific names
all_combinations = []
for gender in profile["gender"]:
    for ethnicity in profile["ethnicity"]:
        # Filter names based on both gender and ethnicity
        gender_specific_names = names_by_gender[gender]
        ethnicity_specific_names = names_by_ethnicity[ethnicity]
        valid_names = list(set(gender_specific_names) & set(ethnicity_specific_names))

        combinations = list(itertools.product(valid_names, profile["age"], [gender], [ethnicity], profile["place_of_birth"], profile["current_residence"]))
        all_combinations.extend(combinations)

# Shuffle randomly all combinations
random.shuffle(all_combinations) 

# Debugging print
print(f"Total combinations before selection: {len(all_combinations)}") # should be 1280


# Select 20 unique combinations ensuring gender balance (half males and half females)
selected_combinations = []
male_count = 0
female_count = 0
for combination in all_combinations:
    if male_count < 10 and combination[2] == 'Male':
        selected_combinations.append(combination)
        male_count += 1
    elif female_count < 10 and combination[2] == 'Female':
        selected_combinations.append(combination)
        female_count += 1
    if male_count == 10 and female_count == 10:
        break


# Debugging print
print(f"Selected combinations: {len(selected_combinations)}") # Should be 20


persona_count = 1
for name, age, gender, ethnicity, birth_place, residence in selected_combinations:

    print(f"==========(Persona No.{persona_count} Creation)==========")

    profile['name'] = name
    profile['age'] = age
    profile['gender'] = gender
    profile["ethnicity"] = ethnicity
    profile['place_of_birth'] = birth_place
    profile['current_residence'] = residence

    profile_messages = []
    profile_messages.append(
        {"role": "user", "content": prompts["ethnicity_prompt"].format(**profile)}
    )

    response = client.chat.completions.create(
        model = "gpt-4", 
        messages = profile_messages,
        temperature = 0.9
    )

    profile_messages.append(response.choices[0].message)
    profile_messages.append(
        {"role": "user", "content": prompts["disease_prompt"].format(**profile)}
    )

    response = client.chat.completions.create(
        model = "gpt-4", 
        messages = profile_messages,
        temperature = 0.9
    )
    profile_messages.append(response.choices[0].message)

    profile_messages.append(
        {"role": "user", "content": prompts["profile_prompt"].format(profile=profile, **profile)}
    )

    response = client.chat.completions.create(
        model = "gpt-4",
        messages = profile_messages,
        temperature = 0.9
    )
    profile_content = response.choices[0].message.content
    print(profile_content)

    pattern = re.compile(r"^\d+\.")
    profile_sections = [
        section.strip()
        for section in profile_content.split("\n\n")
        if pattern.match(section)
    ]

    persona = {
        "profile": profile.copy(),
        "content": profile_content
    }

    # Save the current persona to a JSON file
    file_path = "C:/Users/ntanavarass/Desktop/Conversational-Triple-Extraction/data generation/personas generation/personas.json"
    try:
        with open(file_path, "a+", encoding="utf-8") as file:
            if os.stat(file_path).st_size == 0:
                file.write('[')
            else:
                file.seek(file.tell() - 1)
                file.write(', ')
            json.dump(persona, file, indent=4)
            # file.write(']')
    except Exception as e:
        print(f"Failed to write persona No.{persona_count}: {e}")

    print(f"==========(Persona No.{persona_count} Completion)==========")
    persona_count += 1

print("All personas written successfully.")


Total combinations before selection: 640
Selected combinations: 20
==========(Persona No.1 Creation)==========
1. **Personal and Demographic Background**:
   - Yusuf, a 60-year-old male with a Turkish ethnicity, was born in Amsterdam and now resides in Rotterdam in the Netherlands. He is known for his jovial nature and is fondly called "Joey" by his close friends.
   - He comes from a large family with six siblings, four kids of his own and a grandchild who he dotes on. Yusuf's mother also had Type 2 Diabetes, indicating a family history of the disease.

2. **Physical Characteristics and Health**:
   - Yusuf is 5'10" tall with gray hair and olive skin. He carries a little extra weight, a result of years of indulging in delicious Turkish cuisine and less physical activity.
   - He has a noticeable birthmark on his left cheek and suffers from Type 2 Diabetes, which he is struggling to manage. He often dresses in comfortable clothing and is cautious about his diet due to his medical condi